# TUTOR-60 · 4 — Reporting

The trial has a number. Now it has to reach a board that will not open a notebook, in
three formats, this year and every year after, for the state and for each of its eight
regions — and every one of those documents has to carry the same intervals, the same
definitions and the same provenance as the fit.

`axiom.report` treats that as a **template plus a context**. The template is a `Spec`
with a content hash: it holds the layout, the styling and the names of the numbers, and
no numbers. The context is the data. Rendering is a pure function of the two, into HTML,
PowerPoint or PDF.

| | |
|---|---|
| **1** | build the template, and see that it contains no data |
| **2** | build the context from the fit, and let `missing` say what is not there yet |
| **3** | resolve and render, three formats from one description |
| **4** | run the same template again — a different family, a different theme — and watch the hash |

In [ ]:
import sys

sys.path.insert(0, ".")

import pathlib
import tempfile

import numpy as np
import pandas as pd

import tutoring as T
from axiom.core import LedgerLine, is_failure, summarize
from axiom.infer import PymcBackend
from axiom.report import (
    FORMATS, Format, Report, ReportBuilder, ResolvedFigure, ResolvedLedger, ResolvedMetric,
    ResolvedReport, ResolvedTable, ResolvedText, Theme, missing, placeholders, render_html,
    render_pdf, render_pptx, resolve, write,
)
from axiom.surface import HillKernel, SplineKernel, marginal_band, response_band

from axiom.surface import Surface, fit, predict

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

program = T.run(T.chosen_design())
spline_spec = T.planning_spec()
result = fit(spline_spec, T.school_panel(program),
             backend=PymcBackend(nuts_sampler="nutpie"), draws=1000, tune=1000, chains=4, seed=0)
print("fitted:", spline_spec.name, "| sigma", round(result.posterior.summary("sigma").mean, 2))

## 1 · The template

`ReportBuilder` is a small language for saying what a document contains. Everything in
braces is a placeholder filled from the context; every `metric`, `figure`, `table` and
`ledger` names a **source** rather than carrying a value.

In [ ]:
template = (
    ReportBuilder("tutor60_readout", "TUTOR-60 — dose-response readout")
    .subtitle("{n_schools} schools, one school year. Prepared {as_of}. "
              "Every interval is {interval_label}.")
    .footer("axiom.report · template {template_hash} · fit {fit_hash}")

    .section("What the trial found",
             summary="One year, 120 randomized schools, growth against each school's own prior year.")
    .paragraph("Tutoring raises reading growth steeply to about {peak_minutes:.0f} weekly "
               "minutes and **stops paying** above it. Caregiver messaging adds "
               "{messaging_gain:.1f} points for {messaging_cost:.0f} dollars a student.")
    .metric("pilot_contrast", "Growth at the pilot allocation", unit="points/year")
    .metric("recommended_contrast", "Growth at 35 weekly minutes", unit="points/year")
    .metric("residual_sd", "Residual sd, school level", unit="points/year")

    .section("The response, with its uncertainty",
             summary="Both curves carry a 90 % band, because a curve without one is a claim "
                     "the model did not make.")
    .figure("response", caption="Expected growth against the tutoring allocation")
    .figure("marginal", caption="Marginal effect — where the band crosses zero, the next "
                                "dollar is as likely to hurt as to help")
    .figure("messaging", caption="Caregiver messaging saturates by the second message a week")

    .section("Detail")
    .table("by_band", caption="Schools, students and observed growth by allocation band",
           precision=2)
    .divider()
    .ledger("ledger", caption="What the numbers above rest on", show_detail=True)
    .build()
)
print("template hash :", template.content_hash()[:16])
print("sections      :", [section.title for section in template.sections])
print("sources       :", template.sources())
print("placeholders  :", placeholders(template.subtitle), placeholders(template.footer))

serialized = template.to_json()
print(f"\nthe template is {len(serialized):,} bytes of layout and contains no data:")
print("  a fitted number in it?", any(token in serialized for token in ("6.85", "2.59", "120 ")))
print("  round-trips        :", Report.from_json(serialized) == template)

## 2 · The context, and what is missing from it

`missing` compares the template's sources and placeholders against a context and names
what is absent. Running it against an empty dictionary is the checklist for building one.

In [ ]:
print("everything this template needs:")
table([[name] for name in missing(template, {})], headers=("the template needs",))

In [ ]:
band = response_band(result, "tutoring", n_grid=41, mass=0.9)
slope = marginal_band(result, "tutoring", n_grid=41, mass=0.9)
messaging_band = response_band(result, "messaging", n_grid=16, mass=0.9)
assert not is_failure(band) and not is_failure(slope) and not is_failure(messaging_band)

peak = float(band.doses[int(np.argmax(band.mean))])

def contrast_at(fitted, spec_, dose: float) -> np.ndarray:
    """Posterior draws of growth at an allocation, against no program at all."""
    surface = Surface(spec_)
    high = predict(surface, fitted.posterior, {"tutoring": dose, "messaging": T.MESSAGING_MAX}, seed=0)
    low = predict(surface, fitted.posterior, {"tutoring": 0.0, "messaging": 0.0}, seed=0)
    return (np.asarray(high.values) - np.asarray(low.values)).ravel()

means = program.school_means("growth")
means["band"] = pd.cut(means.tutoring, [-1, 450, 950, 1900, 3001],
                       labels=["0-450", "500-900", "1000-1800", "2000-3000"])
by_band = (means.groupby("band", observed=True)
           .agg(schools=("school", "size"), students=("eligible", "sum"),
                mean_growth=("outcome", "mean"))
           .reset_index()
           .rename(columns={"band": "allocation (USD/student-year)"}))

context: dict[str, object] = {
    "as_of": "2027-06-30",
    "n_schools": program.n_schools,
    "interval_label": band.label(),
    "template_hash": template.content_hash()[:12],
    "fit_hash": spline_spec.content_hash()[:12],
    "peak_minutes": peak / T.MINUTE_COST,
    "messaging_gain": float(messaging_band.mean[-1] - messaging_band.mean[0]),
    "messaging_cost": T.MESSAGING_MAX,
    "pilot_contrast": summarize(contrast_at(result, spline_spec, 1800.0), mass=0.9),
    "recommended_contrast": summarize(contrast_at(result, spline_spec, 700.0), mass=0.9),
    "residual_sd": float(result.posterior.summary("sigma").mean),
    "response": band,
    "marginal": slope,
    "messaging": messaging_band,
    "by_band": by_band,
    "ledger": [
        LedgerLine(kind="identification",
                   statement="schools were randomized to allocations; the adjustment set is empty",
                   detail={"route": "backdoor", "unit_of_assignment": "school"}),
        LedgerLine(kind="estimand",
                   statement="the contrast is against the *assigned* allocation, not the "
                             "delivered one; schools delivered 93% of assigned minutes",
                   detail={"version": "assigned"}),
        LedgerLine(kind="interval",
                   statement=f"curves reported at {band.label()} over {band.n_draws:,} posterior draws"),
        LedgerLine(kind="analysis",
                   statement="six benchmark readings per school averaged before fitting; "
                             "treating them as independent narrows every interval by 70%",
                   detail={"rows_fitted": str(program.n_schools)}),
        LedgerLine(kind="model",
                   statement="natural cubic spline on eight knots — a family that permits the "
                             "response to turn over, chosen before the data",
                   detail={"spec_hash": spline_spec.content_hash()[:16]}),
    ],
}
print("missing now:", missing(template, context) or "nothing")

## 3 · Resolve, then render

`resolve` binds the context into the template and produces a `ResolvedReport` — still a
value, still inspectable, and now carrying the numbers. Rendering happens after that, so
the three formats cannot disagree about what they are rendering.

In [ ]:
resolved = resolve(template, context)
assert isinstance(resolved, ResolvedReport)
table(
    [
        [s.title, len(s.blocks),
         str([type(b).__name__.removeprefix("Resolved") for b in s.blocks])]
        for s in resolved.sections
    ],
    headers=("section", "blocks", "kinds"),
)

metrics = [b for s in resolved.sections for b in s.blocks if isinstance(b, ResolvedMetric)]
figures = [b for s in resolved.sections for b in s.blocks if isinstance(b, ResolvedFigure)]
print("\nmetrics carry their interval:")
table(
    [[m.label, m.text, m.interval_text or "(no interval)"] for m in metrics],
    headers=("metric", "value", "interval"),
)
print("\nevery figure leads with its band:",
      [figure.figure.data[0].fill == "toself" for figure in figures])

In [ ]:
out = pathlib.Path(tempfile.mkdtemp())
written = {}
rows = []
for name in FORMATS:
    chosen: Format = name
    path = write(template, context, str(out / f"tutor60_readout.{name}"), chosen)
    if is_failure(path):
        rows.append([name, "refused", path.reason])
    else:
        written[name] = pathlib.Path(path)
        rows.append([name, f"{written[name].stat().st_size:,} bytes", written[name].name])
table(rows, headers=("format", "written", "file / or refused because"))

html = render_html(resolved)
print(f"\nHTML: {html.count('plotly-graph-div')} interactive figures, "
      f"{html.count(chr(34) + 'fill' + chr(34) + ':' + chr(34) + 'toself' + chr(34))} band traces, "
      f"template hash embedded: {template.content_hash() in html}")
print(f"HTML: the interval definition reached the page: {band.label() in html}")

## 4 · The same template, again

This is the part that makes it a template rather than a document. Re-render with a
different response family — the monotone Hill fit from notebook 3 — and the layout, the
styling and the hash are unchanged while every number moves.

In [ ]:
hill_spec = T.spec({"tutoring": HillKernel(reference_dose=900.0, amplitude_scale=8.0),
                    "messaging": SplineKernel(reference_dose=T.MESSAGING_MAX, amplitude_scale=2.0,
                                              knots=T.MESSAGING_KNOTS)}, name="tutor60_hill")
hill = fit(hill_spec, T.school_panel(program),
           backend=PymcBackend(nuts_sampler="nutpie", target_accept=0.95),
           draws=1500, tune=2000, chains=4, seed=0)
hill_band = response_band(hill, "tutoring", n_grid=41, mass=0.9)
hill_messaging = response_band(hill, "messaging", n_grid=16, mass=0.9)

# The monotone family cannot report a marginal effect at zero dose, and says so rather
# than drawing one: a Hill curve with shape below 1 has infinite slope at the origin, and
# 4 % of the posterior draws are there. The report refuses the figure it cannot make.
refused = marginal_band(hill, "tutoring", n_grid=41, mass=0.9)
print(f"marginal_band from zero -> {type(refused).__name__}")
print(" ", refused.reason)
hill_slope = marginal_band(hill, "tutoring", doses=list(np.linspace(150.0, T.TUTORING_MAX, 40)), mass=0.9)
print(f"\nfrom 150 USD up      -> {type(hill_slope).__name__}, "
      f"shape parameter {hill.posterior.summary('s_tutoring').interval}")

monotone_context = {
    **context,
    "fit_hash": hill_spec.content_hash()[:12],
    "peak_minutes": float(hill_band.doses[int(np.argmax(hill_band.mean))]) / T.MINUTE_COST,
    "messaging_gain": float(hill_messaging.mean[-1] - hill_messaging.mean[0]),
    "pilot_contrast": summarize(contrast_at(hill, hill_spec, 1800.0), mass=0.9),
    "recommended_contrast": summarize(contrast_at(hill, hill_spec, 700.0), mass=0.9),
    "residual_sd": float(hill.posterior.summary("sigma").mean),
    "response": hill_band, "marginal": hill_slope, "messaging": hill_messaging,
    "ledger": [*context["ledger"][:-1],
               LedgerLine(kind="model",
                          statement="Hill kernel — monotone by construction; it cannot report a "
                                    "response that turns over",
                          detail={"spec_hash": hill_spec.content_hash()[:16]})],
}
monotone = resolve(template, monotone_context)
assert isinstance(monotone, ResolvedReport)
print("same template hash:", resolved.source_hash == monotone.source_hash == template.content_hash())
table(
    [
        [left.label, f"{left.text} {left.interval_text or ''}".strip(),
         f"{right.text} {right.interval_text or ''}".strip()]
        for left, right in zip(metrics, [b for s in monotone.sections for b in s.blocks
                                         if isinstance(b, ResolvedMetric)])
    ],
    headers=("metric", "spline", "Hill"),
)
print(f"\npeak of the curve: {context['peak_minutes']:.0f} vs "
      f"{monotone_context['peak_minutes']:.0f} weekly minutes")

The whole report — subtitle, three figures, five ledger lines, a table — regenerates from
a different fit with one dictionary. That is what makes "produce this for each of the
eight regions, every June" a loop rather than a project.

Styling is a value too, so a house theme changes the hash without touching what the
report is *about*.

In [ ]:
house = Theme(
    name="state_department", font="Helvetica", accent_color="#2f5d8a", muted_color="#6b7280",
    palette=("#2f5d8a", "#b5453b", "#3aa17e", "#c9a227"), base_size=10.5, title_size=24.0,
    page="letter", slide="widescreen", figure_height=300.0, band_opacity=0.20,
)
restyled = template.model_copy(update={"theme": house})
print("restyling changes the hash :", restyled.content_hash() != template.content_hash())
print("but not what it asks for   :", restyled.sources() == template.sources())
deck = render_pptx(resolve(restyled, context))
paged = render_pdf(resolve(restyled, context))
table(
    [
        [label, f"{len(made):,} bytes" if isinstance(made, bytes) else made.reason]
        for label, made in (("pptx", deck), ("pdf", paged))
    ],
    headers=("restyled", "result"),
)

## What notebook 4 delivered

1. **A template with no data in it.** The layout, the styling and the names of the
   numbers are a `Spec` with a content hash; `Report.from_json` round-trips it; a grep for
   a fitted number finds nothing.
2. **`missing` as a checklist.** Every source and every placeholder the template needs,
   named, before anything is rendered — and a render against an incomplete context
   returns `Unsupported` rather than a document with a hole in it.
3. **Three formats, one description.** HTML with interactive figures, a slide deck and a
   paged PDF, all from the same `ResolvedReport`, all carrying the same 90 % bands and the
   same ledger.
4. **Repeatability is the point.** The same template rendered against the monotone fit
   changes every number and no structure. That is what makes an annual readout, or eight
   regional ones, a loop.

Notebook 5 uses the fit to make the decision, and fills the recommendation into this same
report.